In [1]:
from esmdmsfunctions import *

/net/dali/home/barton/dhw28/popDMS/esmDMS/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 515/515 [00:01<00:00, 337.57it/s, Materializing param=encoder.layer.29.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/arch

In [3]:
whole_df = get_unique_df()

In [5]:
whole_df["Embeddings"][0].shape[0]

31

In [7]:
whole_df["PreNums"][0].shape[0]

3

In [12]:
selected_layer=12
n_reps = whole_df['PreNums'].iloc[0].shape[0]
        
replicate_dfs = []
for rep in range(n_reps):
    df_rep = whole_df.copy()
    df_rep['PreNums']  = df_rep['PreNums'].apply(lambda x: x[rep])
    df_rep['PostNums'] = df_rep['PostNums'].apply(lambda x: x[rep])
    df_rep = df_rep.drop(columns=['ProteinSequence'])
    replicate_dfs.append(df_rep)

# Only decompose the selected layer — skip the full loop
decomposed_dfs = []
for rep_df in replicate_dfs:
    decomposed_df = rep_df[['PreNums', 'PostNums']].copy()
    decomposed_df[f'Layer{selected_layer}'] = rep_df['Embeddings'].apply(
        lambda x: x[selected_layer]
    )
    decomposed_dfs.append(decomposed_df)

df_selection = pd.DataFrame()
for i, decomposed_df in enumerate(decomposed_dfs):
    df_selection[f'Rep{i+1}_PreNums']  = decomposed_df['PreNums']
    df_selection[f'Rep{i+1}_PostNums'] = decomposed_df['PostNums']
df_selection['Embedding'] = decomposed_dfs[-1][f'Layer{selected_layer}']


In [14]:
embeddings = np.stack(df_selection['Embedding'].values)

In [20]:
dims = embeddings.shape[1]
for dim in range(dims):
    embeddings[:, dim] = z_normalize(embeddings[:, dim])
df_selection['Embedding'] = list(embeddings)

In [22]:
embeddings = np.stack(df_selection['Embedding'].values)

In [2]:
sim_data = get_simulation_results(30, sel_func=gaussian_selection)

Running layer 0...
Running simulation...
(array([ 39,  16,   7, ...,   2, 110,  70], shape=(13172,)), array([ 47,  13,   6, ...,   7, 164,  94], shape=(13172,)), array([53, 18,  6, ...,  0, 94, 64], shape=(13172,)))
  Analyzing generation 30...
Running layer 1...
Running simulation...
(array([ 39,  16,   7, ...,   2, 110,  70], shape=(13172,)), array([ 47,  13,   6, ...,   7, 164,  94], shape=(13172,)), array([53, 18,  6, ...,  0, 94, 64], shape=(13172,)))
  Analyzing generation 30...


KeyboardInterrupt: 

In [4]:
whole_df = get_unique_df()

In [5]:
import sys
for name, obj in list(globals().items()):
    size = sys.getsizeof(obj)
    if size > 100_000:  # only show objects > 1 MB
        print(f"{name}: {size / 1024**2:.1f} MB")

whole_df: 14.4 MB
